# Batch Normalization

This formula represents the **Batch Normalization** operation, a fundamental technique in Deep Learning used to stabilize and accelerate the training of artificial neural networks. 

The standard mathematical representation splits the operation into two equations:

$$\hat{x} = \frac{x - E[x]}{\sqrt{Var[x] + \epsilon}}$$

$$y = \hat{x} \cdot \gamma + \beta$$

Here is the breakdown and explanation of each element in the formula, grouped into two logical stages:

### 1. The Normalization Stage (Centering and Scaling)
The goal of this stage is to transform the input distribution ($x$) to have a mean of 0 and a variance of 1.

* **$x$**: The current input value (the activation of a neuron in a hidden layer for a specific training example).
* **$E[x]$**: The mean (Expected Value) of the inputs $x$ calculated across the current mini-batch of data. Subtracting the mean ($x - E[x]$) centers the distribution around **0**.
* **$Var[x]$**: The variance of the current mini-batch, which measures the spread of the data points. 
* **$\epsilon$ (Epsilon)**: A very small constant (e.g., $10^{-5}$) added to the variance for **numerical stability**. It prevents division-by-zero errors in case the batch variance happens to be exactly 0.
* **$\sqrt{Var[x] + \epsilon}$**: Represents the adjusted standard deviation. Dividing by this value scales the variance of the entire batch to **1**.

### 2. The Scaling and Shifting Stage (Parameter Learning)
If a network only normalized the data, it would limit the representational capacity of the neural layer (for example, it would force inputs to sigmoidal activation functions to stay strictly within their linear region). To fix this, two learnable parameters are introduced:

* **$\gamma$ (Gamma)**: The **scale** parameter. It allows the network to modify the optimal variance if a value other than 1 yields better learning results.
* **$\beta$ (Beta)**: The **shift** (bias) parameter. It allows the network to move the mean away from 0 to a more appropriate value.

Both variables ($\gamma$ and $\beta$) are **learnable parameters** (torch.nn.Parameter). The network learns their optimal values through backpropagation, just like standard weights. If the network determines that normalization harms performance, it can completely undo the transformation by learning $\gamma = \sqrt{Var[x]}$ and $\beta = E[x]$.

Without them, BatchNorm would strictly force every single channel to have a mean of 0 and a variance of 1, severely damaging the network's ability to learn.

---

### Why Is This Formula Important?
1. **Reduces Internal Covariate Shift**: It keeps the distribution of activations stable during training, even when the weights of previous layers change drastically.
2. **Allows Higher Learning Rates**: The network becomes less sensitive to weight initialization and less prone to vanishing or exploding gradients.
3. **Acts as a Regularizer**: Because the mean and variance are calculated over small mini-batches (which vary statistically), it introduces a small amount of noise. This noise helps reduce overfitting.


## BatchNorm Update Equations

In PyTorch, the framework I personally use, the running statistics are updated using an **Exponential Moving Average**.

$$ \hat{\mu}_t = (1 - m) \cdot \hat{\mu}_{t-1} + m \cdot \mu_B $$

$$ \hat{\sigma}^2_t = (1 - m) \cdot \hat{\sigma}^2_{t-1} + m \cdot \sigma_B^2 $$

Where:
* **$\hat{\mu}_t$ / $\hat{\sigma}^2_t$**: The updated running mean and running variance (stored in `running_mean` and `running_var` buffers).
* **$\hat{\mu}_{t-1}$ / $\hat{\sigma}^2_{t-1}$**: The historical running mean and variance from the previous step.
* **$\mu_B$ / $\sigma_B^2$**: The mean and variance calculated strictly from the current mini-batch $B$.
* **$m$**: The `momentum` hyperparameter (default value is `0.1` in PyTorch).

### 2. The $Z$-Score Normalization Formulas

#### During Training (`model.train()`)
The mini-batch is normalized using its own current statistics. The momentum does **not** affect this step; it only updates the background buffers:

$$ X_{norm} = \frac{X - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} $$

#### During Inference (`model.eval()`)
The mini-batch statistics are ignored. The model uses the accumulated running statistics calculated via the momentum equations above:

$$ X_{norm} = \frac{X - \hat{\mu}_t}{\sqrt{\hat{\sigma}^2_t + \epsilon}} $$

*Note: $\epsilon$ (epsilon) is a tiny constant (default `1e-5`) added for numerical stability to prevent division by zero.*